# Actividad 12 — Etapa B: Preparación de la Capa de Consumo Analítico
### Proyecto Integrador — Ingeniería de Datos y Consultas con Spark SQL
**Objetivo:** Describir y justificar el dataset final (Capa Gold), formular tres preguntas analíticas de negocio, implementar consultas reproducibles con Spark SQL / DataFrames y documentar el desacoplamiento entre el cómputo ETL y las consultas analíticas.

---
## 1. Selección y Justificación del Dataset Final (Capa Gold)
El activo de datos consumido es `data/processed/proyecto_integrado_mensual.parquet`, generado por nuestro pipeline modular en PySpark.

### Ficha Técnica del Dataset
* **Ubicación:** `data/processed/proyecto_integrado_mensual.parquet`
* **Granularidad:** 1 fila por mes calendario (`PERIODO = YYYY-MM-01`). Cobertura continua de 187 meses (enero de 2011 a julio de 2026).
* **Calidad:** 0 registros duplicados, 0 valores nulos en todas las columnas.

### Justificación de las 12 Columnas Seleccionadas
1. **`PERIODO` (`date`):** Llave primaria temporal estandarizada al primer día del mes para unir de manera consistente todas las series.
2. **`INGRESOS_REALES` (`double`):** Ingresos del Gobierno Federal (`XBB`) deflactados con el INPC a precios constantes de dic-2025. Permite evaluar la capacidad recaudatoria real.
3. **`GASTOS_REALES` (`double`):** Gasto Total del Gobierno Federal (`XBC`) deflactado con el INPC. Mide el tamaño real del desembolso público sin distorsión inflacionaria.
4. **`INPC` (`double`):** Índice Nacional de Precios al Consumidor (base dic-2025). Insumo para calcular la inflación general interanual de México.
5. **`Total` (`double`):** Índice IGAE general (base 2018=100). Representa el volumen de producción económica agregada del país.
6. **`Actividades primarias` (`double`):** IGAE agropecuario. Permite aislar la dinámica del campo mexicano.
7. **`Actividades secundarias` (`double`):** IGAE industrial y manufacturero. Es el sector clave ligado a las cadenas de exportación e inversión con EE.UU.
8. **`Actividades terciarias` (`double`):** IGAE de servicios y comercio. Refleja el dinamismo del consumo y la demanda doméstica interna.
9. **`TC_CIERRE` (`double`):** Tipo de cambio FIX (USD/MXN) al cierre del último día hábil del mes. Mide el nivel de equilibrio cambiario mensual.
10. **`SP500_CIERRE` (`double`):** Cierre mensual del índice accionario de EE.UU. Proxy de la riqueza y valuación bursátil global.
11. **`SP500_PROMEDIO` (`double`):** Media mensual de cotizaciones diarias del S&P 500. Suaviza la volatilidad intradiaria y se alinea mejor con flujos acumulados mensuales (IGAE/Ingresos).
12. **`VIX_CIERRE` (`double`):** Índice de volatilidad CBOE. Termómetro de incertidumbre y aversión global al riesgo (*fear index*).

### Articulación con los Notebooks de Exploración Previa (`04` y `05`)
* En `04_analisis_tipo_cambio.ipynb` se analizó la sensibilidad del Tipo de Cambio frente a la inflación (pass-through) y las finanzas públicas.
* En `05_analisis_sp500.ipynb` se evaluó el comportamiento del S&P 500 (cierre vs promedio) y el VIX frente al sector secundario e industrial.
* En este notebook (`06_etapa_b_capa_consumo.ipynb`), en lugar de simples correlaciones bivariadas, utilizamos **Spark SQL** para ejecutar agregaciones analíticas multidimensionales que responden a las preguntas centrales de investigación.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pipeline.py").exists():
    os.chdir("..")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Actividad12_EtapaB_CapaConsumo")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

DATA_PATH = "data/processed/proyecto_integrado_mensual.parquet"
df_gold = spark.read.parquet(DATA_PATH)
df_gold.createOrReplaceTempView("macro_mexico")

print(f"Capa Gold cargada: {df_gold.count()} filas x {len(df_gold.columns)} columnas")
df_gold.printSchema()


## 2. Definición de Preguntas Analíticas de Negocio
Alineadas con el objetivo de evaluar la transmisión de choques globales hacia la economía mexicana:

1. **Pregunta 1 (Estrés Financiero Global y Co-movimiento en México):** ¿Cómo se comportaron simultáneamente el balance fiscal real (`INGRESOS_REALES - GASTOS_REALES`), la manufactura mexicana (`Actividades secundarias`) y el tipo de cambio durante los meses en que el mercado global experimentó alta incertidumbre (`VIX_CIERRE >= 25`)?
2. **Pregunta 2 (Evolución Anual y Brecha Fiscal):** ¿Cuál fue el promedio anual y la tasa de crecimiento del Tipo de Cambio, del S&P 500 y de la recaudación tributaria real por cada año del periodo 2011-2026?
3. **Pregunta 3 (Régimen de Mercado y Elasticidad):** Al segmentar la muestra en regímenes de mercado alcista (`S&P 500 > 3000`) vs periodos previos, ¿cambió la proporción de participación de la industria y los servicios en el IGAE total?

## 3. Implementación de Consultas de Consumo con Spark SQL

### Consulta 1: Diagnóstico de Meses de Alta Volatilidad Global (VIX >= 25)

In [ ]:
query_1 = """
SELECT 
    PERIODO,
    ROUND(VIX_CIERRE, 2) AS vix,
    ROUND(TC_CIERRE, 2) AS tipo_cambio,
    ROUND(SP500_PROMEDIO, 1) AS sp500_prom,
    ROUND((INGRESOS_REALES - GASTOS_REALES) / 1000000, 2) AS balance_fiscal_mdp,
    ROUND(`Actividades secundarias`, 2) AS igae_industria,
    ROUND(Total, 2) AS igae_total
FROM macro_mexico
WHERE VIX_CIERRE >= 25.0
ORDER BY VIX_CIERRE DESC
"""

res_1 = spark.sql(query_1)
print("Meses históricos con pánico/alta volatilidad global (VIX >= 25):")
res_1.show(15, truncate=False)


### Consulta 2: Agregaciones Anuales y Tendencia Histórica (Consolidación Anual)

In [ ]:
query_2 = """
SELECT 
    YEAR(PERIODO) AS anio,
    ROUND(AVG(TC_CIERRE), 2) AS tc_promedio_anual,
    ROUND(AVG(SP500_PROMEDIO), 1) AS sp500_promedio_anual,
    ROUND(AVG(VIX_CIERRE), 2) AS vix_promedio_anual,
    ROUND(SUM(INGRESOS_REALES) / 1000000, 1) AS ingresos_totales_anuales_mdp,
    ROUND(SUM(GASTOS_REALES) / 1000000, 1) AS gastos_totales_anuales_mdp,
    ROUND((SUM(INGRESOS_REALES) - SUM(GASTOS_REALES)) / 1000000, 1) AS deficit_superavit_anual_mdp,
    ROUND(AVG(Total), 2) AS igae_promedio_anual
FROM macro_mexico
GROUP BY YEAR(PERIODO)
ORDER BY anio ASC
"""

res_2 = spark.sql(query_2)
print("Consolidado histórico anual de variables integradas:")
res_2.show(20, truncate=False)


### Consulta 3: Segmentación por Regímenes de Volatilidad (Calma vs Estrés Global)

In [ ]:
query_3 = """
SELECT 
    CASE 
        WHEN VIX_CIERRE < 15 THEN '1. Baja Volatilidad (Calma < 15)'
        WHEN VIX_CIERRE BETWEEN 15 AND 25 THEN '2. Volatilidad Media (15 - 25)'
        ELSE '3. Alta Volatilidad (Estrés >= 25)'
    END AS regimen_global,
    COUNT(*) AS total_meses,
    ROUND(AVG(TC_CIERRE), 2) AS tc_promedio,
    ROUND(AVG(`Actividades secundarias`), 2) AS media_igae_industria,
    ROUND(AVG(`Actividades terciarias`), 2) AS media_igae_servicios,
    ROUND(AVG(INGRESOS_REALES - GASTOS_REALES) / 1000000, 2) AS balance_fiscal_medio_mdp
FROM macro_mexico
GROUP BY 1
ORDER BY 1 ASC
"""

res_3 = spark.sql(query_3)
print("Comportamiento de la economía mexicana según el régimen de estrés global:")
res_3.show(truncate=False)


## 4. Documentación: Trabajo ETL vs. Cómputo en Consulta (Write Once vs. Query Time)
Siguiendo los principios de la **Clase 12** sobre desacoplamiento arquitectónico:

| Etapa / Criterio | Lo que se preparó UNA SOLA VEZ en el Pipeline (ETL) | Lo que se calcula ESPECÍFICAMENTE al Consultar (Consumo) |
| :--- | :--- | :--- |
| **Transformaciones** | Limpieza de caracteres (`N/E`), casteo numérico y estandarización de clave mensual `PERIODO`. | Agrupaciones anuales (`YEAR(PERIODO)`), filtros condicionales (`VIX >= 25`) y cálculo de déficit (`INGRESOS - GASTOS`). |
| **Manejo de Granularidad** | Reducción de 8,791 registros diarios de tipo de cambio a 187 meses con `max_by`. | Clasificación por regímenes de estrés (`CASE WHEN VIX...`) y agregación de promedios ponderados. |
| **Integración de Fuentes** | Cruce de 3 fuentes distintas mediante `BroadcastHashJoin`. | Consultas sobre una única tabla lógica (`macro_mexico`), sin necesidad de ejecutar nuevos *joins*. |
| **Costo Computacional** | Costo periódico y pesado de compilación, ordenamiento y persistencia en disco Parquet. | Consultas ultra-rápidas en memoria (`Spark SQL / In-Memory MPP`) con lectura selectiva de columnas (*Projection Pruning*). |